<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/form-8k/filings-by-item.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Finding EDGAR Filings by Item

The following examples show how to search for specific SEC filings based on their items using the Query API. These examples focus on locating 8-K filings that include Item 1.03, "Bankruptcy or Receivership," as well as Form D filings referencing items 6b and 3C.1. The approach can be adapted to search for any EDGAR filings that reference different items.

The EDGAR form types that include or reference specific items and can be searched by item are as follows:

- 8-K
- 8-K12B
- 8-K12G3
- 8F-2 ORDR
- D
- APP ORDR
- APP NTC
- APP WDG
- 1-U
- ABS-15G
- N-8F ORDR
- N-8F NTC
- 6B ORDR

In [ ]:
pip install -q sec-api

In [ ]:
from sec_api import QueryApi

queryApi = QueryApi(api_key="YOUR_API_API_KEY")

### Finding 8-K Filings by Event Items

This example demonstrates how to search for Form 8-K filings based on the event item IDs disclosed within the filing. The URLs and accession numbers of the located filings can then be used in combination with the [Extractor API](https://sec-api.io/docs/sec-filings-item-extraction-api) to extract and download the desired content sections of the event items.

For instance, the following search query retrieves the metadata of 8-K filings that include Item 1.03, "Bankruptcy or Receivership":

```
formType:"8-K" AND items:"1.03"
```

This query filters filings with a form type of "8-K" that also contain Item 1.03. To search for other event items, simply replace `1.03` with the desired item ID. For example, `items:"3.01"` corresponds to the section "Notice of Delisting or Failure to Satisfy a Continued Listing Rule or Standard; Transfer of Listing."

To further refine the search by date, a date range can be added:

```
filedAt:[2019-01-01 TO 2022-12-31]
```

This ensures that filings within the specified date range are retrieved, allowing for more focused results when running the query.

In [ ]:
search_params = {
    "query": 'formType:"8-K" AND items:"1.03" AND filedAt:[2021-01-01 TO 2023-12-31]',
    "from": "0",
    "size": "50",
    "sort": [{"filedAt": {"order": "desc"}}],
}

response = queryApi.get_filings(search_params)

In [ ]:
import pandas as pd

filings = pd.json_normalize(response['filings'])

In [ ]:
print('Total filings matching the search query:', response['total']['value'])
print('Filings returned per API call:', len(filings))

Total filings matching the search query: 282
Filings returned per API call: 50


A total of 282 Form 8-K filings with Item 1.03 were filed between 2021-01-01 and 2023-12-31. The Query API limits the number of filings returned per request to a maximum of 50. In order to retrieve all 282 filings, we need to implement pagination by adjusting the `from` parameter in subsequent requests while keeping the `size` parameter constant at `50`. The `from` parameter indicates the starting position or offset in the search results.

To fetch all 282 Form 8-K filings, we can define a function called `get_filings(query)` that handles the pagination process. This function will retrieve the next set of 50 filings by incrementing the `from` parameter in each subsequent request until no more filings are returned.

In [ ]:
def get_filings(query):
    search_params = {
        "query": query,
        "from": 0,
        "size": "50",
        "sort": [{"filedAt": {"order": "desc"}}],
    }

    all_filings = []

    while True:
        response = queryApi.get_filings(search_params)
        filings = response["filings"]

        if len(filings) == 0:
            break

        all_filings.extend(filings)
        search_params["from"] += 50

    return pd.json_normalize(all_filings)


search_query = 'formType:"8-K" AND items:"1.03" AND filedAt:[2021-01-01 TO 2023-12-31]'

form_8K_with_1_03 = get_filings(search_query)

In [ ]:
print('Number of 8-K filings with Item 1.03 between 2021 and 2023:\n', len(form_8K_with_1_03))

Number of 8-K filings with Item 1.03 between 2021 and 2023:
 282


In [ ]:
print('Metadata of 8-K filings')
print('------------------------')
form_8K_with_1_03[["formType", "accessionNo", "cik", "ticker", "filedAt", "items", "linkToFilingDetails"]].head()

Metadata of 8-K filings
------------------------


,formType,accessionNo,cik,ticker,filedAt,items,linkToFilingDetails
0,8-K,0001193125-23-304154,716006,YELLQ,2023-12-27T17:20:57-05:00,"[Item 1.03: Bankruptcy or Receivership, Item 2...",https://www.sec.gov/Archives/edgar/data/716006...
1,8-K,0001104659-23-129388,84129,RADCQ,2023-12-27T06:03:09-05:00,[Item 1.01: Entry into a Material Definitive A...,https://www.sec.gov/Archives/edgar/data/84129/...
2,8-K/A,0001861449-23-000222,1861449,BRDSQ,2023-12-26T21:05:59-05:00,[Item 1.01: Entry into a Material Definitive A...,https://www.sec.gov/Archives/edgar/data/186144...
3,8-K,0001213900-23-098373,1698113,IDICQ,2023-12-26T09:09:08-05:00,[Item 1.01: Entry into a Material Definitive A...,https://www.sec.gov/Archives/edgar/data/169811...
4,8-K,0001861449-23-000220,1861449,BRDSQ,2023-12-26T08:54:08-05:00,[Item 1.01: Entry into a Material Definitive A...,https://www.sec.gov/Archives/edgar/data/186144...


In [ ]:
print('Items disclosed in the first 2 filings:')
list(form_8K_with_1_03['items'])[:2]

Items disclosed in the first 2 filings:


[['Item 1.03: Bankruptcy or Receivership',
  'Item 2.01: Completion of Acquisition or Disposition of Assets'],
 ['Item 1.01: Entry into a Material Definitive Agreement',
  'Item 1.03: Bankruptcy or Receivership',
  'Item 8.01: Other Events',
  'Item 9.01: Financial Statements and Exhibits']]

The `linkToFilingDetails` field in the response contains the URLs to the 8-K filings and `items` provides the item IDs of the triggering events that are disclosed in each filing. These URLs can be used to extract and download the textual content of Item 1.03 from all the located filings using the [Extractor API](https://sec-api.io/docs/sec-filings-item-extraction-api).

### Find Form D Filings by Item

The next example demonstrates how to find Form D filings by items specified under "Item 6 - Federal Exemptions and Exclusions Claimed". The following list represents commonly used items:

- `04`
- `04.1`
- `04.2`
- `04.3`
- `4a5`
- `06b`
- `06c`
- `3C`
- `3C.1`
- `3C.3`
- `3C.4`
- `3C.5`
- `3C.6`
- `3C.7`
- `3C.9`

The following query will search for Form D filings (`formType:D`) that include both `06b` and `3C.1` items. Modify the query to include other items as needed.

 ```
 formType:D AND items:"06b" AND items:"3C.1"
 ```

In [ ]:
search_query = (
    'formType:D AND items:"06b" AND items:"3C.1" AND filedAt:[2023-01-01 TO 2023-01-31]'
)

form_D_filings = get_filings(search_query)

In [ ]:
print('Number of Form D filings found:', len(form_D_filings))

Number of Form D filings found: 1166


1166 Form D filings were found that match the search criteria. To validate and ensure that all filings include items 6b and 3C.1 in the `items` field, inspect the `items` field of the first five filings using the following code snippet:

In [ ]:
list(form_D_filings['items'])[:5]

[['Item 06b: ',
  'Item 3C: Investment Company Act Section 3(c)',
  'Item 3C.1: Section 3(c)(1)'],
 ['Item 06b: ',
  'Item 3C: Investment Company Act Section 3(c)',
  'Item 3C.1: Section 3(c)(1)'],
 ['Item 06b: ',
  'Item 3C: Investment Company Act Section 3(c)',
  'Item 3C.1: Section 3(c)(1)'],
 ['Item 06b: ',
  'Item 3C: Investment Company Act Section 3(c)',
  'Item 3C.1: Section 3(c)(1)',
  'Item 3C.7: Section 3(c)(7)'],
 ['Item 06b: ',
  'Item 3C: Investment Company Act Section 3(c)',
  'Item 3C.1: Section 3(c)(1)']]